In [ ]:
import sys
from pathlib import Path

def find_project_root(markers=["src", ".env"], max_hops=7):
    """Busca la raíz del proyecto por múltiples marcadores"""
    p = Path.cwd()
    for _ in range(max_hops):
        if any((p / marker).exists() for marker in markers):
            return p
        p = p.parent
    raise RuntimeError(f"No se encontró la raíz del proyecto desde {Path.cwd()}")

# Encontrar y añadir la raíz al path
PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"[OK] Raíz del proyecto: {PROJECT_ROOT}")
print(f"[OK] sys.path actualizado")

In [ ]:
from datetime import date
from pathlib import Path
# Importar después de configurar sys.path
from src.whoscored_viz.whoscored_fixtures import make_driver, scrape_range_finished, FIXTURES_URL
from src.whoscored_viz.paths import BASE_DATA_DIR

COMP_SLUG   = "laliga"
SEASON_SLUG = "2025-2026"

start_date = date(2025, 9, 30)
end_date   = date(2025, 10, 6)

# Usar BASE_DATA_DIR en lugar de PROJECT_ROOT
out_dir = BASE_DATA_DIR / "raw" / "fixtures"

# Verificar que la URL está bien definida
print(f"FIXTURES_URL: {FIXTURES_URL}")
print(f"out_dir: {out_dir}")

driver = make_driver(use_uc=True, headless=False)
try:
    df_fixtures = scrape_range_finished(
        driver, start_date, end_date, out_dir,
        comp_slug=COMP_SLUG, season_slug=SEASON_SLUG, 
        save_json=True, fixtures_url=FIXTURES_URL  # Pasar explícitamente la URL
    )
    print(f"✅ Scraping completado. Filas obtenidas: {len(df_fixtures)}")
    print(f"📁 Datos guardados en: {out_dir}/DataFixtures/{COMP_SLUG}/{SEASON_SLUG}/")
finally:
    driver.quit()

df_fixtures.head() if not df_fixtures.empty else print("No se encontraron partidos finalizados")